<a href="https://colab.research.google.com/github/Giorgio-Colombo/Data-Analysis-Portfolio/blob/main/clash_royale_strategy_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisi Econometrica e Strategica del Meta di Clash Royale (Dataset Giugno 2026)

Questo notebook implementa un'analisi quantitativa rigorosa su scala Big Data (~10 milioni di record) per identificare l'efficacia reale di carte e mazzi competitivi, affrontando i due principali problemi dei dati empirici in ladder:
1. **Disparità di livello delle carte (Pay-to-Win Bias):** Una carta o un mazzo possono registrare un win rate elevato solo perché impiegati con livelli medi superiori rispetto a quelli avversari.
2. **Concentrazione e bias di campionamento (Player Bias):** Un mazzo può apparire dominante semplicemente perché giocato con volumi enormi da un singolo giocatore d'élite.

### Pipeline Metodologica:
- **Ingestione & Ottimizzazione RAM:** Filtraggio pushdown e vettorizzazione ad alta densità.
- **Modello Logistico Differenziale Sparso:** Depurazione dell'effetto dei livelli mediante stima del parametro $\beta_{\text{level}}$.
- **Codifica Binaria a 64-bit:** Raggruppamento e hashing deterministico delle combinazioni di 8 carte.
- **Intervallo di Confidenza di Wilson (95%):** Selezione conservativa dei mazzi con margine di prestazione statisticamente superiore all'atteso.

## Sezione 1: Setup dell'Ambiente e Configurazione Parametri
Vengono inizializzati i moduli di calcolo scientifico e definiti i parametri di inferenza statistica. Viene inoltre scaricato l'archivio Parquet e caricato il vocabolario JSON delle carte.

In [1]:
import os
import gc
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import kagglehub

# ============ CONFIGURAZIONE STATISTICA & AMBIENTE ============
FILE        = '2026-06-21_0.parquet'
MAX_PARTITE = None        # None = processa tutte le partite disponibili
N_MODELLO   = 1_000_000   # Partite campionate per la regressione logistica
MIN_USO     = 5000        # Frequenza minima di una carta nel campione per l'inferenza
MIN_PARTITE = 300         # Soglia campionaria minima per valutare un mazzo completo
MIN_GIOC    = 30          # Numero minimo di utenti distinti per neutralizzare il player bias
MAX_QUOTA   = 0.10        # Quota massima monopolizzabile da un singolo giocatore (<= 10%)
SEED        = 42
# ==============================================================

# 1. Download/Localizzazione Dataset Kaggle
if 'path_2026' not in globals():
    path_2026 = kagglehub.dataset_download('jackmangione/clash-royale-matchups-june2026')
print("Cartella dataset:", path_2026)

# 2. Mappatura Dizionario Carte (ID intero -> Nome Carta)
with open(os.path.join(path_2026, 'cardToID.json'), encoding='utf-8') as f:
    card_to_id = json.load(f)
id_to_card = {int(v): k for k, v in card_to_id.items()}
print(f"Mappa carte caricata con successo: {len(id_to_card)} carte uniche censite.")

100%|██████████| 5.73G/5.73G [01:13<00:00, 84.1MB/s]

Extracting files...


Cartella dataset: /root/.cache/kagglehub/datasets/jackmangione/clash-royale-matchups-june2026/versions/6
Mappa carte caricata con successo: 176 carte uniche censite.


## Sezione 2: Caricamento Dati ed Ottimizzazione della Memoria (Big Data)
In questa fase:
1. Si estraggono unicamente i match competitivi regolari (`game_mode == 'Ladder'`).
2. Gli identificatori testuali dei giocatori (`winner_id`, `loser_id`) vengono fattorizzati in interi a 32-bit (`int32`), riducendo drasticamente l'occupazione di memoria.
3. Vengono identificati e rimossi i duplicati dovuti a interrogazioni concorrenti dell'API tramite la terna univoca `(time, w_pid, l_pid)`.
4. Le variabili vengono serializzate in array NumPy compatti, liberando i DataFrame intermedi dalla memoria.

In [2]:
path = os.path.join(path_2026, FILE)
wc  = [f"winner_card_{i}" for i in range(8)]
lc  = [f"loser_card_{i}" for i in range(8)]
wlv = [f"winner_card_{i}_level" for i in range(8)]
llv = [f"loser_card_{i}_level" for i in range(8)]
cols = ["winner_id", "loser_id", "time", "game_mode"] + wc + lc + wlv + llv

# Lettura Parquet con filtraggio a monte
df = pd.read_parquet(path, columns=cols, filters=[("game_mode", "==", "Ladder")])
df = df.drop(columns="game_mode")
if MAX_PARTITE and len(df) > MAX_PARTITE:
    df = df.sample(MAX_PARTITE, random_state=SEED)
df = df.reset_index(drop=True)
print("Partite Ladder caricate:", f"{len(df):,}")

# Fattorizzazione ID giocatori per comprimere l'occupazione di RAM
n0 = len(df)
codes, _ = pd.factorize(pd.concat([df["winner_id"], df["loser_id"]], ignore_index=True))
df["w_pid"] = codes[:n0].astype(np.int32)
df["l_pid"] = codes[n0:].astype(np.int32)
del codes
df = df.drop(columns=["winner_id", "loser_id"])

# Deduplicazione basata su timestamp e identificatori
dup = df.duplicated(["time", "w_pid", "l_pid"])
print(f"Partite duplicate rimosse: {dup.sum():,}")
df = df[~dup].reset_index(drop=True)

# Conversione in array NumPy compatti e garbage collection
W_all, L_all = df[wc].to_numpy(), df[lc].to_numpy()
lvl_w = df[wlv].to_numpy().mean(axis=1).astype(np.float32)
lvl_l = df[llv].to_numpy().mean(axis=1).astype(np.float32)
w_pid, l_pid = df["w_pid"].to_numpy(), df["l_pid"].to_numpy()
del df; gc.collect()

n_tot = len(W_all)
NC = int(max(W_all.max(), L_all.max())) + 1
assert NC <= 256, "ID carta oltre 255: cambia la codifica dei mazzi"

Partite Ladder caricate: 9,960,921
Partite duplicate rimosse: 0


## Sezione 3: Diagnostica Descrittiva del Campione
Viene quantificato il livello di concentrazione dei giocatori (frequenza degli account dominanti) e calcolato lo squilibrio empirico tra il livello medio dei mazzi vincenti rispetto a quelli perdenti.

In [3]:
pres = np.bincount(np.concatenate([w_pid, l_pid]))
print("=" * 80)
print("DIAGNOSTICA CAMPIONARIA")
print("=" * 80)
print(f"Giocatori unici censiti: {np.count_nonzero(pres):,}")
print(f"Partite del giocatore più attivo: {pres.max():,}")
print(f"Quota presenze dei 100 giocatori più attivi: {np.sort(pres)[-100:].sum() / (2 * n_tot):.3%}")
print(f"Livello medio mazzo: Vincitori {lvl_w.mean():.2f} | Perdenti {lvl_l.mean():.2f}")
print(f"Vantaggio medio grezzo per chi vince: +{lvl_w.mean() - lvl_l.mean():.2f} livelli")

DIAGNOSTICA CAMPIONARIA
Giocatori unici censiti: 6,420,945
Partite del giocatore più attivo: 508
Quota presenze dei 100 giocatori più attivi: 0.252%
Livello medio mazzo: Vincitori 13.47 | Perdenti 13.25
Vantaggio medio grezzo per chi vince: +0.22 livelli


## Sezione 4: Modello Econometrico di Scelta Discreta (Regressione Logistica Sparsa)
Per stimare il valore intrinseco di ciascuna carta, si imposta una regressione logistica binaria con simmetrizzazione casuale dei lati:
$$\text{logit}(P(Y=1)) = X_c \beta_c + \beta_{\text{lvl}} \Delta_{\text{lvl}}$$
Dove:
- **Lati $A$ e $B$:** Assegnati casualmente mediante lancio di moneta (`flip`), rendendo bilanciata la variabile dipendente $Y$.
- **Matrice Differenziale Sparsa $X_c = O_A - O_B$:** Assume valore $+1$ se la carta è presente solo nel mazzo $A$, $-1$ se presente solo nel mazzo $B$, e $0$ se assente o presente in entrambi (*mirror match*, che viene neutralizzato algebricamente).
- **$\beta_{\text{lvl}}$:** Quantifica l'effetto marginale di un singolo livello medio di vantaggio del mazzo sul logit di vittoria.

In [4]:
# 1. Calcolo del Win Rate grezzo per carta sull'intero dataset
vitt  = np.bincount(W_all.ravel(), minlength=NC)
sconf = np.bincount(L_all.ravel(), minlength=NC)
wr_grezzo = np.where(vitt + sconf > 0, vitt / np.maximum(vitt + sconf, 1), np.nan)

# 2. Campionamento e simmetrizzazione A/B casuale
rng = np.random.default_rng(SEED)
idx_s = rng.choice(n_tot, size=min(N_MODELLO, n_tot), replace=False)
n = len(idx_s)
flip = (rng.random(n) < 0.5)                     # True: A = perdente
Ws, Ls = W_all[idx_s], L_all[idx_s]
CA, CB = np.where(flip[:, None], Ls, Ws), np.where(flip[:, None], Ws, Ls)
y = (~flip).astype(int)                          # 1 = A ha vinto
dl = np.where(flip, lvl_l[idx_s] - lvl_w[idx_s], lvl_w[idx_s] - lvl_l[idx_s])[:, None]

# Costruzione matrice One-Hot sparsa per le carte
def onehot(C):
    rows = np.repeat(np.arange(n), 8)
    return sp.csr_matrix((np.ones(n * 8), (rows, C.ravel())), shape=(n, NC))

OA, OB = onehot(CA), onehot(CB)
Xc = (OA - OB).tocsr()
uso = np.asarray((OA + OB).sum(axis=0)).ravel()

# Split Train/Test (80% - 20%)
perm = rng.permutation(n)
tr, te = perm[: int(n * 0.8)], perm[int(n * 0.8):]

def fit(X):
    m = LogisticRegression(C=0.5, fit_intercept=False, max_iter=300)
    m.fit(X[tr], y[tr])
    return m, roc_auc_score(y[te], m.predict_proba(X[te])[:, 1])

# Stima dei tre modelli per confronto
m_c,  auc_c  = fit(Xc)
m_l,  auc_l  = fit(sp.csr_matrix(dl))
m_cl, auc_cl = fit(sp.hstack([Xc, sp.csr_matrix(dl)]).tocsr())
b_lvl = m_l.coef_[0][0]

print("=" * 80)
print("RISULTATI SPECIFICHE DEL MODELLO LOGISTICO")
print("=" * 80)
print(f"ROC-AUC Modello Solo Carte:                 {auc_c:.3f}")
print(f"ROC-AUC Modello Solo Livelli:               {auc_l:.3f}")
print(f"ROC-AUC Modello Completo (Carte + Livelli): {auc_cl:.3f}")
print(f"Effetto marginale di +1 livello medio (logit, b_lvl): {b_lvl:.3f}")

RISULTATI SPECIFICHE DEL MODELLO LOGISTICO
ROC-AUC Modello Solo Carte:                 0.650
ROC-AUC Modello Solo Livelli:               0.665
ROC-AUC Modello Completo (Carte + Livelli): 0.711
Effetto marginale di +1 livello medio (logit, b_lvl): 0.844


## Sezione 5: Diagnostica delle Carte (Coefficienti Depurati vs Grezzi)
Vengono confrontati i coefficienti stimati controllando per i livelli con quelli ottenuti ignorando il livello delle carte. Viene inoltre calcolata la correlazione di rango di Spearman tra i due ordinamenti per verificare l'entità delle distorsioni.

In [5]:
res = pd.DataFrame({
    "carta_id": np.arange(NC),
    "nome_carta": [id_to_card.get(c, f"Card_{c}") for c in range(NC)],
    "uso": uso,
    "wr_grezzo": wr_grezzo,
    "coef_senza_livelli": m_c.coef_[0],
    "coef_con_livelli": m_cl.coef_[0][:NC],
})
res["variazione"] = res["coef_con_livelli"] - res["coef_senza_livelli"]
r = res[res["uso"] >= MIN_USO].copy()

print("=" * 80)
print("ANALISI DI RANGO SULLE CARTE (Soglia d'uso >= 5.000)")
print("=" * 80)
print(f"Numero di carte confrontate: {len(r)}")
print("Correlazione Spearman senza vs con livelli:", round(spearmanr(r["coef_senza_livelli"], r["coef_con_livelli"])[0], 3))
print("Correlazione Spearman win rate grezzo vs coef. corretto:", round(spearmanr(r["wr_grezzo"], r["coef_con_livelli"])[0], 3))

cc = ["carta_id", "nome_carta", "uso", "wr_grezzo", "coef_senza_livelli", "coef_con_livelli", "variazione"]

print("\n--- TOP 10 CARTE CON IL MAGGIOR BIAS DA LIVELLO (Variazione Assoluta) ---")
display(r.reindex(r["variazione"].abs().sort_values(ascending=False).index).head(10)[cc])

print("\n--- TOP 10 CARTE PIÙ EFFICACI (Coefficiente Depurato dai Livelli) ---")
display(r.sort_values("coef_con_livelli", ascending=False).head(10)[cc])

ANALISI DI RANGO SULLE CARTE (Soglia d'uso >= 5.000)
Numero di carte confrontate: 171
Correlazione Spearman senza vs con livelli: 0.943
Correlazione Spearman win rate grezzo vs coef. corretto: 0.749

--- TOP 10 CARTE CON IL MAGGIOR BIAS DA LIVELLO (Variazione Assoluta) ---


,carta_id,nome_carta,uso,wr_grezzo,coef_senza_livelli,coef_con_livelli,variazione
103,103,freeze,94561.0,0.441472,-0.129323,0.013055,0.142378
135,135,evo princess,11188.0,0.584489,0.278891,0.140548,-0.138343
172,172,hero mega minion,22318.0,0.581499,0.378031,0.249309,-0.128722
166,166,hero musketeer,79405.0,0.566389,0.221290,0.109298,-0.111992
118,118,goblin curse,35300.0,0.384497,-0.194590,-0.083701,0.110890
74,74,electro spirit,83401.0,0.508972,-0.045308,0.061744,0.107053
116,116,royal delivery,62596.0,0.509973,-0.001885,0.104686,0.106571
164,164,hero giant,24210.0,0.494128,0.084301,-0.017477,-0.101778
10,10,skeletons,219923.0,0.543485,0.018771,0.117903,0.099132
114,114,heal spirit,16282.0,0.355960,-0.289385,-0.190840,0.098545



--- TOP 10 CARTE PIÙ EFFICACI (Coefficiente Depurato dai Livelli) ---


,carta_id,nome_carta,uso,wr_grezzo,coef_senza_livelli,coef_con_livelli,variazione
175,175,hero barbarian barrel,30664.0,0.580749,0.490592,0.417670,-0.072922
160,160,evo zap,25411.0,0.525098,0.349420,0.321725,-0.027696
143,143,evo royal recruits,19248.0,0.534608,0.396018,0.314697,-0.081320
165,165,hero balloon,68897.0,0.555390,0.264752,0.251407,-0.013345
172,172,hero mega minion,22318.0,0.581499,0.378031,0.249309,-0.128722
161,161,evo giant snowball,19332.0,0.503414,0.250667,0.239693,-0.010974
163,163,hero goblins,33982.0,0.539927,0.224511,0.231798,0.007287
170,170,hero bowler,85365.0,0.519645,0.148063,0.224203,0.076139
109,109,the log,561593.0,0.565116,0.207205,0.220564,0.013359
147,147,evo skeleton barrel,62025.0,0.506011,0.162399,0.216975,0.054576


## Sezione 6: Codifica Binaria a 64-bit ed Inferenza sui Mazzi
Per processare milioni di combinazioni senza colli di bottiglia computazionali:
1. I mazzi di 8 carte vengono ordinati e mappati in scalari a 64 bit (`uint64`), consentendo raggruppamenti istantanei in memoria.
2. Per ogni mazzo viene calcolata la **probabilità attesa di vittoria dai soli livelli**:
   $$p_{\text{atteso}} = \frac{1}{1 + e^{-b_{\text{lvl}} \cdot \Delta_{\text{lvl}}}}$$
3. L'incertezza campionaria del win rate reale è stimata tramite l'**intervallo di confidenza al 95% di Wilson** per proporzioni binomiali.
4. La metrica decisionale centrale è il **margine conservativo**:
   $$\text{Margine} = \text{IC}_{\text{basso}} - p_{\text{atteso}}$$
   Un mazzo con $\text{Margine} > 0$ garantisce statisticamente (al 95% di confidenza) una prestazione superiore a quella attesa dai livelli anche nel peggiore scenario campionario.

In [6]:
def codice(M):
    S = np.ascontiguousarray(np.sort(M, axis=1).astype(np.uint8))
    return S.view(np.uint64).ravel()

def nomi(cod):
    ids = np.array([cod], dtype=np.uint64).view(np.uint8)
    return " | ".join(id_to_card.get(int(i), str(int(i))) for i in ids)

# Costruzione archivio unificato match
ap = pd.DataFrame({
    "deck":  np.concatenate([codice(W_all), codice(L_all)]),
    "vinto": np.concatenate([np.ones(n_tot, np.int8), np.zeros(n_tot, np.int8)]),
    "pid":   np.concatenate([w_pid, l_pid]),
    "dlev":  np.concatenate([lvl_w - lvl_l, lvl_l - lvl_w]),
})

# Probabilità teorica attesa dai soli livelli
ap["p_att"] = (1 / (1 + np.exp(-b_lvl * ap["dlev"].to_numpy()))).astype(np.float32)

# Filtro sul volume minimo di partite
cnt = ap["deck".strip()].value_counts()
validi = cnt[cnt >= MIN_PARTITE].index
sub = ap[ap["deck"].isin(validi)]
print(f"Mazzi distinti: {len(cnt):,} | Mazzi con almeno {MIN_PARTITE} partite: {len(validi):,}")

# Aggregazione per mazzo
agg = sub.groupby("deck").agg(
    partite=("vinto", "size"),
    vittorie=("vinto", "sum"),
    atteso=("p_att", "mean"),
    livello_rel=("dlev", "mean")
)

# Calcolo diversificazione giocatori (neutralizzazione player bias)
gp = sub.groupby(["deck", "pid"]).size()
agg["giocatori"] = gp.groupby(level=0).size()
agg["quota_top"] = gp.groupby(level=0).max() / gp.groupby(level=0).sum()

# Calcolo Intervallo di Confidenza di Wilson (95%)
agg["wr"] = agg["vittorie"] / agg["partite"]
z, nn, p = 1.96, agg["partite"], agg["wr"]
centro = (p + z**2 / (2 * nn)) / (1 + z**2 / nn)
mezzo = z * np.sqrt(p * (1 - p) / nn + z**2 / (4 * nn**2)) / (1 + z**2 / nn)
agg["ic_basso"], agg["ic_alto"] = centro - mezzo, centro + mezzo

# Metriche di sovraperformance netta
agg["wr_extra"] = agg["wr"] - agg["atteso"]
agg["margine"] = agg["ic_basso"] - agg["atteso"]      # Margine conservativo > 0

def tabella(d):
    return pd.DataFrame({
        "partite": d["partite"].astype(int),
        "giocatori": d["giocatori"].astype(int),
        "quota_top_%": (d["quota_top"] * 100).round(1),
        "wr_%": (d["wr"] * 100).round(1),
        "atteso_%": (d["atteso"] * 100).round(1),
        "extra_pp": (d["wr_extra"] * 100).round(1),
        "IC95_%": [f"{a*100:.1f}-{b*100:.1f}" for a, b in zip(d["ic_basso"], d["ic_alto"])],
        "liv_rel": d["livello_rel"].round(2),
        "carte": [nomi(c) for c in d.index],
    }).reset_index(drop=True)

Mazzi distinti: 6,389,317 | Mazzi con almeno 300 partite: 2,277


## Sezione 7: Tabelle di Sintesi Finale e Identificazione della Strategia Ottima
Vengono prodotte le tabelle di sintesi a larghezza piena:
1. **I 10 Mazzi più popolari:** Gli archetipi che dominano la quota di mercato in ladder, con la quota di partite detenuta dal giocatore più frequente.
2. **I 15 Mazzi più robusti ed efficienti:** I mazzi che soddisfano i criteri di affidabilità ($\ge 30$ giocatori, quota top player $\le 10\%$) e massimizzano il margine conservativo sopra il win rate atteso dai livelli.

In [7]:
# Visualizzazione senza troncamenti di colonna
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    print("=" * 90)
    print("TOP 10 MAZZI PIÙ USATI IN LADDER (Con concentrazione del giocatore principale)")
    print("=" * 90)
    display(tabella(agg.sort_values("partite", ascending=False).head(10)))

    # Applicazione filtri di affidabilità per la strategia ottima
    aff = agg[(agg["giocatori"] >= MIN_GIOC) & (agg["quota_top"] <= MAX_QUOTA)]
    print("\n" + "=" * 90)
    print(f"TOP 15 MAZZI STRATEGICAMENTE ROBUSTI (Margine Positivo vs Livelli)")
    print(f"Mazzi affidabili censiti: {len(aff)} su {len(agg)}")
    print(f"Mazzi stabilmente sopra l'atteso (limite inferiore IC > atteso): {(aff['margine'] > 0).sum()}")
    print("=" * 90)
    display(tabella(aff.sort_values("margine", ascending=False).head(15)))

TOP 10 MAZZI PIÙ USATI IN LADDER (Con concentrazione del giocatore principale)


,partite,giocatori,quota_top_%,wr_%,atteso_%,extra_pp,IC95_%,liv_rel,carte
0,89176,28458,0.1,43.7,46.599998,-2.9,43.4-44.1,-0.18,knight | archers | giant | minions | musketeer | mini p.e.k.k.a | fireball | arrows
1,49040,20011,0.1,19.3,42.900002,-23.6,18.9-19.6,-0.38,skeletons | musketeer | hog rider | ice spirit | ice golem | cannon | fireball | the log
2,48677,18437,0.1,60.9,49.700001,11.2,60.5-61.3,-0.01,skeletons | electro spirit | x-bow | fireball | the log | evo archers | evo tesla | hero knight
3,32611,16821,0.2,21.5,44.599998,-23.1,21.0-21.9,-0.29,knight | princess | ice spirit | goblin gang | inferno tower | rocket | goblin barrel | the log
4,28233,10258,0.1,59.5,52.400002,7.1,58.9-60.1,0.12,princess | ice spirit | goblin gang | inferno tower | rocket | the log | evo goblin barrel | hero knight
5,27856,11121,0.1,58.8,51.400002,7.4,58.2-59.4,0.07,skeletons | hog rider | ice spirit | fireball | the log | evo musketeer | evo cannon | hero ice golem
6,26216,6346,1.9,10.0,46.299999,-36.3,9.6-10.4,-0.18,balloon | bomber | spear goblins | royal giant | fire spirit | bats | rage | vines
7,25096,8753,0.1,61.5,54.599998,6.9,60.9-62.1,0.23,princess | goblin gang | inferno tower | rocket | the log | evo ice spirit | evo goblin barrel | hero knight
8,23233,8706,0.1,58.8,53.000000,5.8,58.1-59.4,0.15,musketeer | hog rider | ice spirit | fireball | the log | evo skeletons | evo cannon | hero ice golem
9,23023,8985,0.1,56.7,52.400002,4.4,56.1-57.4,0.12,skeletons | hog rider | ice spirit | fireball | the log | evo cannon | hero musketeer | hero ice golem



TOP 15 MAZZI STRATEGICAMENTE ROBUSTI (Margine Positivo vs Livelli)
Mazzi affidabili censiti: 2191 su 2277
Mazzi stabilmente sopra l'atteso (limite inferiore IC > atteso): 629


,partite,giocatori,quota_top_%,wr_%,atteso_%,extra_pp,IC95_%,liv_rel,carte
0,647,312,4.6,91.2,48.099998,43.1,88.8-93.1,-0.11,knight | archers | giant | minions | mini p.e.k.k.a | spear goblins | fireball | arrows
1,345,185,4.1,92.5,49.200001,43.3,89.2-94.8,-0.06,knight | archers | goblins | giant | minions | musketeer | mini p.e.k.k.a | arrows
2,529,254,4.3,92.4,50.500000,42.0,89.9-94.4,0.01,knight | goblins | giant | minions | musketeer | mini p.e.k.k.a | fireball | arrows
3,699,347,2.6,92.4,52.599998,39.9,90.2-94.2,0.13,knight | archers | goblins | giant | minions | musketeer | mini p.e.k.k.a | fireball
4,1030,475,2.5,92.6,54.000000,38.7,90.9-94.1,0.20,knight | archers | goblins | giant | musketeer | mini p.e.k.k.a | fireball | arrows
5,406,175,4.2,91.1,51.599998,39.5,88.0-93.5,0.08,knight | archers | goblins | giant | minions | mini p.e.k.k.a | fireball | arrows
6,350,170,2.9,87.4,49.599998,37.8,83.5-90.5,-0.02,knight | archers | giant | minions | musketeer | mini p.e.k.k.a | goblin hut | arrows
7,1343,664,1.9,85.9,50.299999,35.6,84.0-87.7,0.01,knight | giant | minions | musketeer | mini p.e.k.k.a | spear goblins | fireball | arrows
8,991,458,3.1,87.6,51.799999,35.8,85.4-89.5,0.08,knight | archers | giant | minions | musketeer | mini p.e.k.k.a | goblin hut | fireball
9,1758,921,1.4,88.1,52.900002,35.2,86.5-89.5,0.14,knight | archers | giant | musketeer | mini p.e.k.k.a | spear goblins | fireball | arrows
